# CXR-Recall — Phase 2: contrastive dual-tower (image+text) pretraining

Separate notebook from `cxr_recall_arm1_kaggle.ipynb` (Phase 1: image-only Arms 1/2/3, and Phase 2 attempt 1: the flawed end-to-end fusion that hit modality collapse) — kept apart so each notebook stays focused and a Kaggle session doesn't need to rebuild all of Phase 1 from scratch just to run Phase 2.

Rebuilds Phase 2 the way the published literature actually does image+text (ConVIRT/CLIP/GLoRIA/BioViL/CheXzero-style): separate image and text towers, each projected into a shared 512-dim embedding space, aligned with a contrastive loss — not label classification from text directly. Mirrors Phase 1's Arm 1/2/3 structure so results are directly comparable.

**Phase 1's and Phase 2 attempt 1's real numbers are hardcoded as reference constants below (Section 4)** rather than recomputed — they're already real, documented results (see `RESULTS.md`), and re-running all of Phase 1 (four separate training loops: Arm 1, Arm 2, Arm 3c, Arm 3c-frozen) here just to regenerate numbers already confirmed reproducible via `SEED=42` across three independent runs would waste Kaggle GPU time for no new information.

**Attach the same dataset as the Phase 1 notebook** (+ Add Data → "Indiana University chest x-ray" / "Chest X-rays Indiana University"). Also turn on **Settings → Accelerator → GPU** and **Settings → Internet: on** (`Bio_ClinicalBERT` downloads from the HuggingFace Hub on first use).

In [ ]:
import torch
import random
import numpy as np

print('torch:', torch.__version__, ' cuda available:', torch.cuda.is_available())

SEED = 42

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
print('seed set to', SEED)

## 1. Locate the attached dataset

In [ ]:
import os

INPUT_DIR = '/kaggle/input'
candidates = os.listdir(INPUT_DIR)
assert candidates, "No dataset attached -- use '+ Add Data' and search for the Indiana University chest x-ray collection first."
print('attached datasets:', candidates)

DATASET_DIR = os.path.join(INPUT_DIR, candidates[0])
print('using:', DATASET_DIR)
for root, dirs, files in os.walk(DATASET_DIR):
    print(root, '->', dirs, files[:5])

In [ ]:
import glob

csv_files = glob.glob(os.path.join(DATASET_DIR, '**', '*.csv'), recursive=True)
png_dirs = sorted(set(os.path.dirname(p) for p in glob.glob(os.path.join(DATASET_DIR, '**', '*.png'), recursive=True)))
print('csv files:', csv_files)
print('folders containing pngs:', png_dirs)

## 2. Build the manifest
Adjust the `*_COL` constants below if column names differ from these guesses (the common `raddar/chest-xrays-indiana-university` mirror uses `uid`, `findings`, `impression`, `Problems` in the reports CSV, and `uid`, `filename` in the projections CSV) -- same manifest logic as the Phase 1 notebook's Section 2.

In [ ]:
import pandas as pd

reports_csv = [f for f in csv_files if 'report' in os.path.basename(f).lower()]
projections_csv = [f for f in csv_files if 'projection' in os.path.basename(f).lower()]
assert reports_csv and projections_csv, 'Could not auto-find reports/projections CSVs -- inspect csv_files above and set paths manually.'

reports_df = pd.read_csv(reports_csv[0])
proj_df = pd.read_csv(projections_csv[0])

UID_COL, FINDINGS_COL, IMPRESSION_COL, TAGS_COL = 'uid', 'findings', 'impression', 'Problems'
PROJ_UID_COL, FILENAME_COL = 'uid', 'filename'

merged = proj_df.merge(reports_df, left_on=PROJ_UID_COL, right_on=UID_COL, suffixes=('', '_report'))

def build_report_text(row):
    parts = [str(row.get(FINDINGS_COL, '')), str(row.get(IMPRESSION_COL, ''))]
    return ' '.join(p for p in parts if p and p.lower() != 'nan').strip()

def build_tags(row):
    raw = row.get(TAGS_COL, '')
    if pd.isna(raw):
        return ''
    sep = ';' if ';' in str(raw) else ','
    return ';'.join(t.strip() for t in str(raw).split(sep) if t.strip())

manifest = pd.DataFrame({
    'image_path': merged[FILENAME_COL],
    'patient_id': merged[UID_COL].astype(str),
    'report_text': merged.apply(build_report_text, axis=1),
    'mesh_tags': merged.apply(build_tags, axis=1),
})

IMAGE_ROOT = png_dirs[0]
print('manifest rows:', len(manifest), ' image root:', IMAGE_ROOT)
manifest.head()

## 3. Label vocabulary + patient-level split
Same logic as the Phase 1 notebook's Sections 3-4 -- `SEED=42` reproduces the identical patient-level split, so `train_df`/`val_df`/`label_vocab` here match Phase 1's exactly.

In [ ]:
from collections import Counter

def build_mesh_vocab(tag_lists, top_k=14):
    counts = Counter(tag for tags in tag_lists for tag in tags)
    return [tag for tag, _ in counts.most_common(top_k)]

def mesh_tags_to_vector(tags, vocab):
    tag_set = set(tags)
    return np.array([1.0 if v in tag_set else 0.0 for v in vocab], dtype=np.float32)

def patient_level_split(df, patient_id_col, ratios=(0.8, 0.1, 0.1), seed=42):
    patients = sorted(df[patient_id_col].unique())
    rng = np.random.default_rng(seed)
    rng.shuffle(patients)
    n = len(patients)
    n_train, n_val = int(n * ratios[0]), int(n * ratios[1])
    train_p = set(patients[:n_train])
    val_p = set(patients[n_train:n_train + n_val])
    test_p = set(patients[n_train + n_val:])
    return (df[df[patient_id_col].isin(train_p)],
            df[df[patient_id_col].isin(val_p)],
            df[df[patient_id_col].isin(test_p)])

train_df, val_df, test_df = patient_level_split(manifest, 'patient_id')
print('split sizes:', len(train_df), len(val_df), len(test_df))

train_tags = [t.split(';') if t else [] for t in train_df['mesh_tags']]
label_vocab = build_mesh_vocab(train_tags, top_k=14)
print('label vocab:', label_vocab)

## 4. Phase 1 + Phase 2 attempt 1 reference numbers (hardcoded, not recomputed) + fresh lexical-overlap check

Phase 1's numbers are the seeded run (`SEED=42`) from `RESULTS.md` "Phase 1e" -- the project's best-supported result, confirmed by three independent runs agreeing on direction. Phase 2 attempt 1's numbers are from `cxr_recall_arm1_kaggle.ipynb` Section 11 (the naive end-to-end `MultimodalFusionClassifier`, diagnosed with modality collapse). Neither is recomputed here -- both require training runs (four separate loops for Phase 1 alone) that would cost real GPU time to regenerate numbers already real and documented.

The lexical-overlap diagnostic *is* recomputed fresh below -- it's a pure pandas computation over `train_df`/`label_vocab`, effectively free, so there's no reason to hardcode it.

In [ ]:
# Phase 1 (image-only), seeded run (SEED=42), RESULTS.md "Phase 1e" -- three
# independent runs (two unseeded, one seeded) all agree on direction.
PHASE1_ARM1_AUROC = 0.7092
PHASE1_ARM3C_FROZEN_AUROC = 0.7353
PHASE1_LABEL_AGREEMENT_AT_5 = 0.2727
PHASE1_RANDOM_BASELINE_AGREEMENT = 0.1845

# Phase 2 attempt 1 (naive end-to-end image+text fusion, MultimodalFusionClassifier,
# cxr_recall_arm1_kaggle.ipynb Section 11) -- real numbers, diagnosed as modality
# collapse: image-only ablation barely above chance, worse than Arm 1 alone.
PHASE2_ATTEMPT1_FULL_AUROC = 0.9958
PHASE2_ATTEMPT1_IMAGE_ONLY_AUROC = 0.5328
PHASE2_ATTEMPT1_TEXT_ONLY_AUROC = 0.9958

print('Phase 1 reference:  Arm 1 =', PHASE1_ARM1_AUROC, ' Arm 3c-frozen =', PHASE1_ARM3C_FROZEN_AUROC)
print('Phase 2 attempt 1 reference:  full =', PHASE2_ATTEMPT1_FULL_AUROC,
      ' image_only =', PHASE2_ATTEMPT1_IMAGE_ONLY_AUROC, ' text_only =', PHASE2_ATTEMPT1_TEXT_ONLY_AUROC)

In [ ]:
# How much of any text-modality signal would be circular? Recomputed fresh
# (cheap, pandas-only) rather than hardcoded, since it costs nothing and this
# notebook shouldn't rely on a stale number if train_df/label_vocab ever change.
def label_lexical_overlap(df, vocab):
    total, present = 0, 0
    for _, row in df.iterrows():
        text = str(row['report_text']).lower()
        tags = row['mesh_tags'].split(';') if row['mesh_tags'] else []
        for tag in tags:
            if tag in vocab:
                total += 1
                if tag.lower() in text:
                    present += 1
    return present / total if total else float('nan')

overlap = label_lexical_overlap(train_df, label_vocab)
print(f'label-vocab tags whose own text appears literally in their report: {overlap:.1%} (naive substring floor)')

## 5. Setup: device, batch size, loss, metric

In [ ]:
import copy
import torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torchvision.models import vit_b_16, ViT_B_16_Weights
from sklearn.metrics import roc_auc_score

transform = ViT_B_16_Weights.DEFAULT.transforms()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE, LR = 16, 1e-4
EPOCHS_REAL, PATIENCE = 20, 5
criterion = nn.BCEWithLogitsLoss()

def macro_auroc(y_true, y_score):
    scores = []
    for i in range(y_true.shape[1]):
        if len(np.unique(y_true[:, i])) < 2:
            continue
        scores.append(roc_auc_score(y_true[:, i], y_score[:, i]))
    return float(np.mean(scores)) if scores else float('nan')

def held_out_auroc(forward_fn, loader, label_pos):
    """One pass over a held-out loader with the model already in eval() mode.
    forward_fn(batch_tuple) -> logits; label_pos = index of the label tensor in
    the batch tuple. Used for TEST numbers: val is what every arm's early
    stopping selects on, so val AUROC is an optimistically-selected number;
    test is evaluated exactly once per arm, on the reloaded best checkpoint,
    and never influences any training or selection decision."""
    all_labels, all_scores = [], []
    with torch.no_grad():
        for batch in loader:
            all_scores.append(torch.sigmoid(forward_fn(batch)).cpu().numpy())
            all_labels.append(batch[label_pos].numpy())
    return macro_auroc(np.concatenate(all_labels), np.concatenate(all_scores))

print('setup complete -- device:', device, ' batch size:', BATCH_SIZE, ' lr:', LR)

## 6. Retrieval utilities (function definitions only)

Same `build_index`/`search`/`label_agreement_at_k`/`mean_reciprocal_rank` as the Phase 1 notebook's Section 7 -- needed for this notebook's own Arm 2/Arm 3 below.

In [ ]:
!pip install -q faiss-cpu

In [ ]:
import faiss

def build_index(embeddings):
    embeddings = embeddings.astype(np.float32).copy()
    faiss.normalize_L2(embeddings)
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)
    return index

def search(index, queries, k):
    queries = queries.astype(np.float32).copy()
    faiss.normalize_L2(queries)
    return index.search(queries, k)

def label_agreement_at_k(query_labels, retrieved_labels):
    if retrieved_labels.shape[0] == 0:
        return float('nan')
    shares = (retrieved_labels * query_labels).sum(axis=1) > 0
    return float(shares.mean())

def mean_reciprocal_rank(query_labels, retrieved_labels):
    shares = (retrieved_labels * query_labels).sum(axis=1) > 0
    hits = np.nonzero(shares)[0]
    return 0.0 if len(hits) == 0 else 1.0 / (hits[0] + 1)

K = 5
print('retrieval utilities defined -- k =', K)

## 7. Stage 1: dual-tower contrastive pretraining

Phase 2 attempt 1 (Section 4's `PHASE2_ATTEMPT1_*` constants, from `cxr_recall_arm1_kaggle.ipynb` Section 11) revealed two real problems: full fusion AUROC = 0.9958, text-only ablation = 0.9958 (identical), image-only ablation = 0.5328 (barely above chance, *worse* than Arm 1's own 0.7092) — diagnosed as **modality collapse**, on top of a 64.0% lexical-overlap floor between MeSH tags and report text (recomputed fresh above). This section rebuilds Phase 2 the way the published literature actually does image+text (ConVIRT/CLIP/GLoRIA/BioViL/CheXzero-style): separate image and text towers, each projected into a shared 512-dim embedding space, aligned with a contrastive loss — not label classification from text directly. No label-prediction gradient exists during this stage, so there's nothing for one modality to starve the other over.

**What this does and doesn't fix:** removes the modality-collapse failure mode structurally (Section 8 below freezes both towers, so neither can degrade during classifier training). It does **not** by itself resolve the report-text/MeSH circularity risk — a frozen text tower faithfully representing the report's full content can still let a downstream head exploit the same lexical shortcut, just without damaging the image branch. That risk gets re-measured (not re-argued) once Arm 1 has real numbers, in Section 8.

Mirrors Phase 1's Arm 1/2/3 structure exactly, so the project can answer two things: (1) does retrieval-evidence fusion improve classification — Arm 1 vs Arm 3, within each phase; (2) does adding text improve decision-making — same arm, across phases. Retrieval (Arm 2/3 below) stays **image-only**, same FAISS mechanism as Phase 1, just on these new embeddings — isolates exactly one variable (the image encoder's pretraining objective) between phases.

In [ ]:
class IUXrayPairsDataset(Dataset):
    """Stage-1 pretraining dataset: (image, report_text, patient_id), no
    labels -- mirrors src/data/iu_xray.py's IUXrayDataset(mode='pairs'),
    which has existed since early in the project but was never exercised
    until now. train_df/val_df already carry report_text/patient_id
    (Section 2's manifest), no new data prep needed."""
    def __init__(self, df, image_root, transform):
        self.df = df.reset_index(drop=True)
        self.image_root = image_root
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(os.path.join(self.image_root, row['image_path'])).convert('RGB')
        image = self.transform(image)
        text = '' if pd.isna(row['report_text']) else str(row['report_text'])
        return image, text, row['patient_id']

In [ ]:
import math
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

class TextEncoder(nn.Module):
    def __init__(self, model_name='emilyalsentzer/Bio_ClinicalBERT'):
        super().__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.embed_dim = self.model.config.hidden_size

    # max_length=192 (was 256): BERT attention memory scales with sequence
    # length squared, so this cuts that memory by roughly 44% while only
    # truncating 0.9% of IU X-ray reports (checked against the real token
    # length distribution: p99=186, max=365, median=64) -- helps Stage 1 fit
    # a larger contrastive batch on a single Kaggle GPU.
    def forward(self, texts, device, max_length=192):
        tokens = self.tokenizer(texts, padding=True, truncation=True,
                                 max_length=max_length, return_tensors='pt').to(device)
        output = self.model(**tokens)
        return output.last_hidden_state[:, 0, :]  # [CLS] embedding


class ProjectionHead(nn.Module):
    def __init__(self, in_dim, out_dim=512):
        super().__init__()
        self.proj = nn.Linear(in_dim, out_dim)

    def forward(self, x):
        return F.normalize(self.proj(x), dim=-1)  # unit norm


class DualTowerContrastiveModel(nn.Module):
    def __init__(self, embed_dim=512, pretrained=True, text_model_name='emilyalsentzer/Bio_ClinicalBERT'):
        super().__init__()
        weights = ViT_B_16_Weights.DEFAULT if pretrained else None
        self.image_backbone = vit_b_16(weights=weights)
        self.image_backbone.heads = nn.Identity()
        self.text_encoder = TextEncoder(text_model_name)
        self.image_proj = ProjectionHead(768, embed_dim)
        self.text_proj = ProjectionHead(self.text_encoder.embed_dim, embed_dim)
        # CLIP convention: store log-scale, exponentiate at use time; clamped
        # after each optimizer step below (ln(100), CLIP's published bound)
        self.logit_scale = nn.Parameter(torch.tensor(math.log(1 / 0.07)))

    def encode_image(self, images):
        return self.image_proj(self.image_backbone(images))

    def encode_text(self, texts, device):
        return self.text_proj(self.text_encoder(texts, device))

    def forward(self, images, texts, device):
        return self.encode_image(images), self.encode_text(texts, device)


def clip_contrastive_loss(image_embeds, text_embeds, logit_scale):
    # symmetric InfoNCE; requires batch_size >= 2 (no negatives otherwise --
    # a degenerate, silently near-zero loss regardless of alignment quality)
    batch_size = image_embeds.shape[0]
    if batch_size < 2:
        raise ValueError(f'clip_contrastive_loss requires batch_size >= 2, got {batch_size}')
    logits_per_image = logit_scale.exp() * image_embeds @ text_embeds.t()
    targets = torch.arange(batch_size, device=image_embeds.device)
    return (F.cross_entropy(logits_per_image, targets) + F.cross_entropy(logits_per_image.t(), targets)) / 2

print('Stage 1 model/loss defined')

In [ ]:
set_seed(SEED)

from torch.amp import autocast, GradScaler

# Defensive cleanup: if this cell is being re-run in the same kernel session
# after a previous OOM (rather than a fresh restart), the old model_stage1/
# optimizer_s1 from the failed attempt may still be resident on the GPU and
# compound the next attempt's memory pressure. Free them first if they exist.
for _stale_name in ('model_stage1', 'optimizer_s1', 'scaler'):
    if _stale_name in globals():
        del globals()[_stale_name]
torch.cuda.empty_cache()

# Contrastive learning needs many negatives per step to be informative --
# BATCH_SIZE=16 (fine for Arm 1's supervised classification loss) gives only
# 15 negatives per positive pair, likely why the first run's recall@1 came
# back at exact random chance (0.0014, n=736) despite train_loss dropping
# smoothly: the model was overfitting to a nearly-uninformative contrastive
# task, not learning real cross-modal alignment. A dedicated, larger batch
# for Stage 1 only (Arm 1/2/3 below keep BATCH_SIZE=16 -- they don't need
# contrastive negatives).
#
# History: STAGE1_BATCH_SIZE=64 (no AMP) OOM'd (only 116.81 MiB free of the
# 192 MiB needed -- a near-miss). 32 fit and gave a real improvement
# (recall@1 moved off exact chance, Arm 1 image-only jumped 0.5745->0.6600)
# -- the confirmed-working, currently-used value. Added mixed precision
# (autocast + GradScaler) and tried pushing back up to 96, then 64 -- both
# OOM'd (AMP's real savings were smaller than the ~40-50% textbook estimate
# here, likely because Adam's optimizer state doesn't shrink under autocast,
# only activations do). Settled back on 32 rather than continuing to chase a
# larger batch -- AMP is kept anyway for the speed benefit even at the same
# batch size.
# NOTE (2026-09-06): the HEADLINE Phase 2 numbers in RESULTS.md (full 0.8045, image-only
# 0.7042, evidence gain +0.0129) came from a run with STAGE1_BATCH_SIZE = 64 (+AMP, which is
# on below). The audit rerun used 32 and reproduced the weaker batch-32 numbers. Set 64 here
# to get held-out TEST numbers for the headline configuration; 32 is kept as the OOM-safe
# default only.
STAGE1_BATCH_SIZE = 32

train_pairs_ds = IUXrayPairsDataset(train_df, IMAGE_ROOT, transform)
val_pairs_ds = IUXrayPairsDataset(val_df, IMAGE_ROOT, transform)
# drop_last=True: avoids a stray small final batch, which clip_contrastive_loss rejects below batch_size 2
# and which would give weak negatives even above that at STAGE1_BATCH_SIZE's scale
train_loader_pairs = DataLoader(train_pairs_ds, batch_size=STAGE1_BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
val_loader_pairs = DataLoader(val_pairs_ds, batch_size=STAGE1_BATCH_SIZE, shuffle=False, num_workers=2, drop_last=True)

model_stage1 = DualTowerContrastiveModel().to(device)
scaler = GradScaler('cuda')

# ClinicalBERT gets a lower LR than the rest, same justification as the
# Phase 1 notebook's Section 11.
LR_TEXT = 2e-5

# A dedicated lower LR + higher patience for the image tower specifically
# (LR_STAGE1_IMAGE=1e-5, PATIENCE_STAGE1=10) was tried and REVERTED after a
# real run: val contrastive loss still peaked at epoch 3 and then climbed
# steadily and accelerating for 7 more epochs with no sign of recovery
# (0.45 train_loss by epoch 10 vs. 4.36 val_loss, textbook overfitting) --
# the same epoch-3 ceiling as every prior run, just with more wasted compute
# proving it again. This is a real negative result, not an assumption: the
# LR/patience change didn't move the boundary, so it's most likely a
# property of the dataset size (5977 pairs) rather than something this
# specific lever can fix. Reverted to the shared LR/PATIENCE below, which is
# what actually produced the real numbers documented in RESULTS.md.
text_params = list(model_stage1.text_encoder.parameters())
other_params = [p for p in model_stage1.parameters() if not any(p is tp for tp in text_params)]
optimizer_s1 = torch.optim.Adam([
    {'params': other_params, 'lr': LR},
    {'params': text_params, 'lr': LR_TEXT},
])

best_val_loss_s1, best_state_s1, epochs_without_improvement_s1 = float('inf'), None, 0

for epoch in range(1, EPOCHS_REAL + 1):
    model_stage1.train()
    total_loss, train_n = 0.0, 0
    for images, texts, _pids in train_loader_pairs:
        images = images.to(device)
        optimizer_s1.zero_grad()
        with autocast('cuda'):
            image_emb, text_emb = model_stage1(images, texts, device)
            loss = clip_contrastive_loss(image_emb, text_emb, model_stage1.logit_scale)
        scaler.scale(loss).backward()
        scaler.step(optimizer_s1)
        scaler.update()
        model_stage1.logit_scale.data.clamp_(0, math.log(100))  # CLIP's published bound -- safe here regardless of
        total_loss += loss.item() * images.size(0)               # whether scaler.step() actually fired this iteration,
        train_n += images.size(0)                                 # since clamping an in-bounds value is a no-op
    total_loss /= train_n

    model_stage1.eval()
    val_loss_total, val_n = 0.0, 0
    with torch.no_grad(), autocast('cuda'):
        for images, texts, _pids in val_loader_pairs:
            images = images.to(device)
            image_emb, text_emb = model_stage1(images, texts, device)
            val_loss_total += clip_contrastive_loss(image_emb, text_emb, model_stage1.logit_scale).item() * images.size(0)
            val_n += images.size(0)
    val_loss = val_loss_total / val_n
    print(f'[Stage 1] epoch {epoch}/{EPOCHS_REAL}  train_loss={total_loss:.4f}  val_loss={val_loss:.4f}')

    if val_loss < best_val_loss_s1:
        best_val_loss_s1, best_state_s1, epochs_without_improvement_s1 = val_loss, copy.deepcopy(model_stage1.state_dict()), 0
    else:
        epochs_without_improvement_s1 += 1
        if epochs_without_improvement_s1 >= PATIENCE:
            print(f'no val improvement for {PATIENCE} epochs -- stopping early at epoch {epoch}')
            break

model_stage1.load_state_dict(best_state_s1)
print(f'best Stage 1 val_loss = {best_val_loss_s1:.4f}')
torch.save({'model_state_dict': model_stage1.state_dict()}, '/kaggle/working/stage1_dual_tower_best.pt')

In [ ]:
# Optional pretraining health check -- NOT Arm 2/3 (cross-modal retrieval is
# explicitly out of scope for this project's confirmed design, image-only
# retrieval is built in Section 9). Just checks whether Stage 1 learned
# anything at all before spending Arm 1/2/3 compute: for each val image, is
# its own report the closest text embedding in the batch of val pairs (and
# vice versa)? Skippable.
model_stage1.eval()
val_image_embs_diag, val_text_embs_diag = [], []
with torch.no_grad(), autocast('cuda'):
    for images, texts, _pids in val_loader_pairs:
        images = images.to(device)
        img_emb, txt_emb = model_stage1(images, texts, device)
        val_image_embs_diag.append(img_emb.float().cpu())
        val_text_embs_diag.append(txt_emb.float().cpu())
val_image_embs_diag = torch.cat(val_image_embs_diag)
val_text_embs_diag = torch.cat(val_text_embs_diag)

sims = val_image_embs_diag @ val_text_embs_diag.t()
n = sims.shape[0]
i2t_recall1 = (sims.argmax(dim=1) == torch.arange(n)).float().mean().item()
t2i_recall1 = (sims.argmax(dim=0) == torch.arange(n)).float().mean().item()
print(f'[Stage 1 diagnostic, not an Arm] image->text recall@1 = {i2t_recall1:.4f}  '
      f'text->image recall@1 = {t2i_recall1:.4f}  (n={n}, random baseline ~= {1/n:.4f})')

## 8. Phase 2 Arm 1 — frozen linear probe + decomposed ablation

Stage 1's towers are frozen from here on (confirmed decision: Stage 2 never fine-tunes, so neither modality can degrade during classifier training the way the image branch collapsed in Phase 2 attempt 1). A single caching pass extracts both modalities' embeddings once; these same cached arrays get reused again in Sections 9 and 10 (Arm 2, Arm 3) — no repeated ViT/BERT forward passes after this cell.

In [ ]:
class IUXrayMultimodalDatasetP2(Dataset):
    """(image, report_text, label) triple, for caching both modalities'
    embeddings against real labels."""
    def __init__(self, df, image_root, transform, label_vocab):
        self.df = df.reset_index(drop=True)
        self.image_root = image_root
        self.transform = transform
        self.label_vocab = label_vocab

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(os.path.join(self.image_root, row['image_path'])).convert('RGB')
        image = self.transform(image)
        text = '' if pd.isna(row['report_text']) else str(row['report_text'])
        tags = row['mesh_tags'].split(';') if row['mesh_tags'] else []
        label = mesh_tags_to_vector(tags, self.label_vocab)
        return image, text, label


for p in model_stage1.parameters():
    p.requires_grad = False
model_stage1.eval()

@torch.no_grad()
def extract_dual_embeddings(model, loader, device):
    model.eval()
    img_embs, txt_embs, labels_out = [], [], []
    with autocast('cuda'):
        for images, texts, labels in loader:
            images = images.to(device)
            img_embs.append(model.encode_image(images).float().cpu().numpy())
            txt_embs.append(model.encode_text(texts, device).float().cpu().numpy())
            labels_out.append(labels.numpy())
    return np.concatenate(img_embs), np.concatenate(txt_embs), np.concatenate(labels_out)

train_ds_p2 = IUXrayMultimodalDatasetP2(train_df, IMAGE_ROOT, transform, label_vocab)
val_ds_p2 = IUXrayMultimodalDatasetP2(val_df, IMAGE_ROOT, transform, label_vocab)
test_ds_p2 = IUXrayMultimodalDatasetP2(test_df, IMAGE_ROOT, transform, label_vocab)  # held out: never used for selection
train_loader_seq_p2 = DataLoader(train_ds_p2, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
val_loader_p2 = DataLoader(val_ds_p2, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader_p2 = DataLoader(test_ds_p2, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print('caching Stage-1 embeddings for train/val/test (one pass, serves Arm 1, 2, and 3 below)...')
train_img_p2, train_txt_p2, train_lbl_p2 = extract_dual_embeddings(model_stage1, train_loader_seq_p2, device)
val_img_p2, val_txt_p2, val_lbl_p2 = extract_dual_embeddings(model_stage1, val_loader_p2, device)
test_img_p2, test_txt_p2, test_lbl_p2 = extract_dual_embeddings(model_stage1, test_loader_p2, device)
print('cached shapes:', train_img_p2.shape, train_txt_p2.shape, val_img_p2.shape, val_txt_p2.shape, test_img_p2.shape, test_txt_p2.shape)
# Persist the frozen embeddings: every experiment below (probes, retrieval, Arm 3) runs in
# seconds on these arrays, so with these files Stage 1 (~40 min) never has to be repeated
# for a probe-side change. Download from /kaggle/working after the run.
np.savez_compressed('/kaggle/working/phase2_stage1_embeddings.npz',
                    train_img=train_img_p2, train_txt=train_txt_p2, train_lbl=train_lbl_p2,
                    val_img=val_img_p2, val_txt=val_txt_p2, val_lbl=val_lbl_p2,
                    test_img=test_img_p2, test_txt=test_txt_p2, test_lbl=test_lbl_p2,
                    stage1_batch_size=STAGE1_BATCH_SIZE, label_vocab=np.array(label_vocab))
print('saved /kaggle/working/phase2_stage1_embeddings.npz')

In [ ]:
class FrozenDualTowerClassifier(nn.Module):
    """Takes precomputed embeddings, not raw images/text -- no encoder
    inside this class at all. Ablation zeroes a modality's contribution at
    the embedding level, before fusion. With use_gating=False (default),
    this is mathematically exact (no intervening biased layer), verified in
    the local unit tests against head(other_embedding_alone) directly. With
    use_gating=True, a learned scalar gate convex-combines the two terms
    instead of summing them -- built because full fusion landed BELOW
    text-only alone in two separate real runs (a fixed unweighted sum can't
    down-weight a noisier modality per example). THE EXACT-EQUALITY ABLATION
    GUARANTEE DOES NOT HOLD under gating: head is affine (has a bias), and
    the gate is itself a learned function of both inputs including the
    zeroed one -- see src/models/frozen_dual_tower_classifier.py's docstring
    and its tests for the exact relationship gating produces instead."""
    def __init__(self, num_labels, embed_dim=512, use_gating=False):
        super().__init__()
        self.head = nn.Linear(embed_dim, num_labels)
        self.use_gating = use_gating
        if use_gating:
            self.gate = nn.Linear(embed_dim * 2, 1)

    def forward(self, image_embeds, text_embeds, use_image=True, use_text=True):
        if not use_image and not use_text:
            raise ValueError('at least one modality must be used')
        batch, embed_dim = image_embeds.size(0), image_embeds.size(1)
        dev = image_embeds.device
        image_term = image_embeds if use_image else torch.zeros(batch, embed_dim, device=dev)
        text_term = text_embeds if use_text else torch.zeros(batch, embed_dim, device=dev)
        if self.use_gating:
            g = torch.sigmoid(self.gate(torch.cat([image_term, text_term], dim=-1)))
            fused = g * image_term + (1 - g) * text_term
        else:
            fused = image_term + text_term
        return self.head(fused)


# Probe training recipe. History of this line: the shared EPOCHS_REAL=20 cap left every early
# Phase 2 probe undertrained; a first fix (lr=1e-4, 400 epochs, patience 25) STILL left the
# full / gated / text-only probes climbing at the cap (the notebook's own WARNING fired), and the
# image-only probe stopped on a plateau at 0.6958 val when its true optimum is ~0.708 --
# lr=1e-4 is far too small for a linear head on unit-norm 512-d embeddings. Checked locally on
# saved embeddings: lr=1e-2 early-stops after 80-200 epochs for all four heads and reproduces
# sklearn's convex L-BFGS solution to within ~0.002 val AUROC (text 0.9437 vs 0.9423, image 0.7075
# vs 0.7078, full 0.9406 vs 0.9395). So: higher LR, long budget, and the cap WARNING below is a
# hard signal -- if it fires, the printed number is a lower bound, not a result.
LR_PROBE, EPOCHS_PROBE, PATIENCE_PROBE = 1e-2, 3000, 50

def train_arm1(model, train_loader, val_loader, tag, use_image=True, use_text=True,
               epochs=None, patience=None, verbose_every=10):
    """Shared training loop for every probe below. use_image/use_text=False trains a
    DEDICATED single-modality probe (that modality's embedding is zeroed in training AND
    validation), which is the fair no-retrieval baseline for Arm 3 -- unlike the
    ablation of a head trained with both modalities present."""
    epochs = epochs or EPOCHS_PROBE
    patience = patience or PATIENCE_PROBE
    optimizer = torch.optim.Adam(model.parameters(), lr=LR_PROBE)
    best_auroc, best_state, best_epoch, epochs_without_improvement = -1.0, None, 0, 0

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        for img_emb, txt_emb, labels in train_loader:
            img_emb, txt_emb, labels = img_emb.to(device), txt_emb.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(img_emb, txt_emb, use_image=use_image, use_text=use_text), labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * img_emb.size(0)
        total_loss /= len(train_loader.dataset)

        model.eval()
        all_labels, all_scores = [], []
        with torch.no_grad():
            for img_emb, txt_emb, labels in val_loader:
                probs = torch.sigmoid(model(img_emb.to(device), txt_emb.to(device), use_image=use_image, use_text=use_text)).cpu().numpy()
                all_labels.append(labels.numpy())
                all_scores.append(probs)
        val_auroc = macro_auroc(np.concatenate(all_labels), np.concatenate(all_scores))
        if epoch % verbose_every == 0 or epoch == 1:
            print(f'[{tag}] epoch {epoch}/{epochs}  train_loss={total_loss:.4f}  val_macro_auroc={val_auroc:.4f}')

        if val_auroc > best_auroc:
            best_auroc, best_state, best_epoch, epochs_without_improvement = val_auroc, copy.deepcopy(model.state_dict()), epoch, 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f'[{tag}] no val improvement for {patience} epochs -- stopping early at epoch {epoch}')
                break
    else:
        print(f'[{tag}] WARNING: hit the {epochs}-epoch cap without early stopping -- still undertrained, raise EPOCHS_PROBE')

    model.load_state_dict(best_state)
    print(f'best {tag} val_macro_auroc = {best_auroc:.4f}  (epoch {best_epoch})')
    return model, best_auroc


train_ds_arm1_p2 = TensorDataset(torch.from_numpy(train_img_p2), torch.from_numpy(train_txt_p2), torch.from_numpy(train_lbl_p2))
val_ds_arm1_p2 = TensorDataset(torch.from_numpy(val_img_p2), torch.from_numpy(val_txt_p2), torch.from_numpy(val_lbl_p2))
train_loader_arm1_p2 = DataLoader(train_ds_arm1_p2, batch_size=BATCH_SIZE, shuffle=True)
val_loader_arm1_p2 = DataLoader(val_ds_arm1_p2, batch_size=BATCH_SIZE, shuffle=False)
test_ds_arm1_p2 = TensorDataset(torch.from_numpy(test_img_p2), torch.from_numpy(test_txt_p2), torch.from_numpy(test_lbl_p2))
test_loader_arm1_p2 = DataLoader(test_ds_arm1_p2, batch_size=BATCH_SIZE, shuffle=False)

# Ungated (use_gating=False, the default): this is the exact continuity
# anchor for the two already-documented Kaggle runs in RESULTS.md.
set_seed(SEED)
model_arm1_p2 = FrozenDualTowerClassifier(num_labels=len(label_vocab)).to(device)
model_arm1_p2, best_auroc_p2 = train_arm1(model_arm1_p2, train_loader_arm1_p2, val_loader_arm1_p2, 'Phase 2 Arm 1 (ungated)')
torch.save({'model_state_dict': model_arm1_p2.state_dict(), 'label_vocab': label_vocab}, '/kaggle/working/phase2_arm1_best.pt')

# Gated (use_gating=True): cheap to train alongside -- Arm 1 trains only a
# tiny head on already-cached 512-dim embeddings, no encoder forward passes,
# so a second model adds negligible cost. This is the actual test of whether
# a learned gate closes the "full fusion below text-only" gap.
set_seed(SEED)
model_arm1_p2_gated = FrozenDualTowerClassifier(num_labels=len(label_vocab), use_gating=True).to(device)
model_arm1_p2_gated, best_auroc_p2_gated = train_arm1(model_arm1_p2_gated, train_loader_arm1_p2, val_loader_arm1_p2, 'Phase 2 Arm 1 (gated)')
torch.save({'model_state_dict': model_arm1_p2_gated.state_dict(), 'label_vocab': label_vocab}, '/kaggle/working/phase2_arm1_gated_best.pt')

# Dedicated single-modality probes -- the clean baselines. The ablation numbers in the
# next cell (zero one modality inside the FULL head) are kept for continuity with the
# earlier runs, but a head trained with text present has weights tuned for image+text,
# so its image-only ablation understates what the image embedding alone can do. Arm 3
# (Section 10) is compared against model_arm1_p2_img, not against the ablation.
set_seed(SEED)
model_arm1_p2_img = FrozenDualTowerClassifier(num_labels=len(label_vocab)).to(device)
model_arm1_p2_img, best_auroc_p2_img = train_arm1(model_arm1_p2_img, train_loader_arm1_p2, val_loader_arm1_p2, 'Phase 2 Arm 1 (image-only probe)', use_text=False)
set_seed(SEED)
model_arm1_p2_txt = FrozenDualTowerClassifier(num_labels=len(label_vocab)).to(device)
model_arm1_p2_txt, best_auroc_p2_txt = train_arm1(model_arm1_p2_txt, train_loader_arm1_p2, val_loader_arm1_p2, 'Phase 2 Arm 1 (text-only probe)', use_image=False)

In [ ]:
def ablation_eval(model, val_loader):
    full_scores, image_only_scores, text_only_scores, y_true = [], [], [], []
    model.eval()
    with torch.no_grad():
        for img_emb, txt_emb, labels in val_loader:
            img_emb_d, txt_emb_d = img_emb.to(device), txt_emb.to(device)
            full_scores.append(torch.sigmoid(model(img_emb_d, txt_emb_d, use_image=True, use_text=True)).cpu().numpy())
            image_only_scores.append(torch.sigmoid(model(img_emb_d, txt_emb_d, use_image=True, use_text=False)).cpu().numpy())
            text_only_scores.append(torch.sigmoid(model(img_emb_d, txt_emb_d, use_image=False, use_text=True)).cpu().numpy())
            y_true.append(labels.numpy())
    y_true = np.concatenate(y_true)
    return (macro_auroc(y_true, np.concatenate(full_scores)),
            macro_auroc(y_true, np.concatenate(image_only_scores)),
            macro_auroc(y_true, np.concatenate(text_only_scores)))


auroc_full_p2, auroc_image_only_p2, auroc_text_only_p2 = ablation_eval(model_arm1_p2, val_loader_arm1_p2)
test_full_p2, test_image_only_p2, test_text_only_p2 = ablation_eval(model_arm1_p2, test_loader_arm1_p2)  # held out

print('=== Phase 2 Arm 1 (frozen dual-tower, contrastively pretrained): image vs. text vs. fusion ===')
print('(val = selected on by early stopping; TEST = held out, evaluated once on the reloaded best checkpoint)')
print(f'Full (image+text)         val_macro_auroc = {auroc_full_p2:.4f}   TEST = {test_full_p2:.4f}')
print(f'Image-only (text zeroed)  val_macro_auroc = {auroc_image_only_p2:.4f}   TEST = {test_image_only_p2:.4f}')
print(f'Text-only (image zeroed)  val_macro_auroc = {auroc_text_only_p2:.4f}   TEST = {test_text_only_p2:.4f}')

def probe_auroc(model, loader, use_image=True, use_text=True):
    model.eval()
    return held_out_auroc(lambda b: model(b[0].to(device), b[1].to(device), use_image=use_image, use_text=use_text), loader, label_pos=2)

test_imgprobe_p2 = probe_auroc(model_arm1_p2_img, test_loader_arm1_p2, use_text=False)
test_txtprobe_p2 = probe_auroc(model_arm1_p2_txt, test_loader_arm1_p2, use_image=False)
print()
print('--- dedicated single-modality probes (trained on that modality alone; the clean baselines) ---')
print(f'Image-only probe          val_macro_auroc = {best_auroc_p2_img:.4f}   TEST = {test_imgprobe_p2:.4f}')
print(f'Text-only probe           val_macro_auroc = {best_auroc_p2_txt:.4f}   TEST = {test_txtprobe_p2:.4f}')
print(f'(ablation-of-full vs dedicated probe, image: {auroc_image_only_p2:.4f} vs {best_auroc_p2_img:.4f} val -- the gap is how much the fused head was tuned away from images)')
print()
print(f'For reference, Phase 2 attempt 1 (naive end-to-end fusion): '
      f'full={PHASE2_ATTEMPT1_FULL_AUROC:.4f}  image_only={PHASE2_ATTEMPT1_IMAGE_ONLY_AUROC:.4f}  text_only={PHASE2_ATTEMPT1_TEXT_ONLY_AUROC:.4f}')
if auroc_image_only_p2 > PHASE2_ATTEMPT1_IMAGE_ONLY_AUROC + 0.1:
    print(f'Modality collapse appears RESOLVED: image-only ablation went from {PHASE2_ATTEMPT1_IMAGE_ONLY_AUROC:.4f} (attempt 1) '
          f'to {auroc_image_only_p2:.4f} (frozen contrastive) -- the image branch no longer collapses toward chance.')
else:
    print(f'Image-only ablation ({auroc_image_only_p2:.4f}) has NOT clearly improved over attempt 1 ({PHASE2_ATTEMPT1_IMAGE_ONLY_AUROC:.4f}) '
          '-- worth investigating before trusting this redesign resolved the collapse.')
print()
print(f'label-vocab lexical overlap (circularity floor, unchanged by this redesign): {overlap:.1%}')
print('Modality collapse and circularity are DIFFERENT risks: freezing the towers structurally rules out')
print('collapse (no gradient path back into the encoders during this stage), but a frozen text tower can')
print('still let this linear head exploit the same lexical shortcut without any encoder degradation --')
print('this number does not improve just because collapse is fixed, and should be read alongside these results.')

print()
print('=== Phase 2 Arm 1 (GATED fusion): image vs. text vs. fusion ===')
print('CAVEAT: with gating, "image-only"/"text-only" no longer means the same thing as above -- the')
print('exact-equality guarantee (ablated output == head(other embedding alone)) does NOT hold once a')
print('learned gate sits between the embeddings and the head (head is affine, and the gate is itself a')
print('function of both inputs, including the zeroed one). These numbers show how the GATE behaves when')
print('handed a zero vector for one modality, not a clean single-modality model -- read them as such.')
auroc_full_p2_gated, auroc_image_only_p2_gated, auroc_text_only_p2_gated = ablation_eval(model_arm1_p2_gated, val_loader_arm1_p2)
test_full_p2_gated, test_image_only_p2_gated, test_text_only_p2_gated = ablation_eval(model_arm1_p2_gated, test_loader_arm1_p2)
print(f'Full (image+text, gated)         val_macro_auroc = {auroc_full_p2_gated:.4f}   TEST = {test_full_p2_gated:.4f}')
print(f'Image-only (text zeroed, gated)  val_macro_auroc = {auroc_image_only_p2_gated:.4f}   TEST = {test_image_only_p2_gated:.4f}')
print(f'Text-only (image zeroed, gated)  val_macro_auroc = {auroc_text_only_p2_gated:.4f}   TEST = {test_text_only_p2_gated:.4f}')
print()
# The only uncorrupted test of the gate is gated-full vs ungated-full on REAL paired
# inputs -- the gated ablation numbers above are the gate's out-of-distribution response
# to a zero vector it never saw in training, so 'gated full beats gated single-modality'
# is not evidence of anything (the run documented in RESULTS.md was first misread that way).
if test_full_p2_gated > test_full_p2:
    print(f'Gated full fusion beats ungated full fusion on TEST ({test_full_p2_gated:.4f} vs {test_full_p2:.4f}) -- '
          'the gate helps on real paired inputs.')
else:
    print(f'Gated full fusion does NOT beat ungated full fusion on TEST ({test_full_p2_gated:.4f} vs {test_full_p2:.4f}) -- '
          'the gate is a regression or a wash on real paired inputs; ignore the gated ablation rows for this question.')
print(f'(val: gated {auroc_full_p2_gated:.4f} vs ungated {auroc_full_p2:.4f})')


## 9. Phase 2 Arm 2 — retrieval reuse (image-only, contrastively pretrained embeddings)

Zero new retrieval mechanism — literally `build_index`/`search`/`label_agreement_at_k`/`mean_reciprocal_rank` (Section 6), called on `train_img_p2`/`val_img_p2` instead of Phase 1's 768-dim ImageNet embeddings. Retrieval stays image-only by design (confirmed decision) — isolates the image encoder's pretraining objective as the only variable changed vs. Phase 1's Arm 2.

In [ ]:
index_p2 = build_index(train_img_p2)
val_scores_p2, neighbor_idx_p2 = search(index_p2, val_img_p2, K)

agreements_p2 = [label_agreement_at_k(val_lbl_p2[i], train_lbl_p2[neighbor_idx_p2[i]]) for i in range(len(val_img_p2))]
mrrs_p2 = [mean_reciprocal_rank(val_lbl_p2[i], train_lbl_p2[neighbor_idx_p2[i]]) for i in range(len(val_img_p2))]
print(f'[Phase 2 Arm 2] label_agreement@{K}={np.mean(agreements_p2):.4f}  MRR={np.mean(mrrs_p2):.4f}')

test_scores_p2, test_neighbor_idx_p2 = search(index_p2, test_img_p2, K)  # test queries retrieve from the TRAIN index only
test_agreements_p2 = [label_agreement_at_k(test_lbl_p2[i], train_lbl_p2[test_neighbor_idx_p2[i]]) for i in range(len(test_img_p2))]
test_mrrs_p2 = [mean_reciprocal_rank(test_lbl_p2[i], train_lbl_p2[test_neighbor_idx_p2[i]]) for i in range(len(test_img_p2))]
print(f'[Phase 2 Arm 2 TEST] label_agreement@{K}={np.mean(test_agreements_p2):.4f}  MRR={np.mean(test_mrrs_p2):.4f}')
print(f'[Phase 1 Arm 2, for comparison] label_agreement@{K}={PHASE1_LABEL_AGREEMENT_AT_5:.4f}')
print(f'[Phase 1 Arm 2 random baseline, for comparison] label_agreement@{K}={PHASE1_RANDOM_BASELINE_AGREEMENT:.4f}')
if np.mean(agreements_p2) > PHASE1_LABEL_AGREEMENT_AT_5:
    print('Contrastive pretraining produced an easier-to-search image space than ImageNet-only pretraining.')
else:
    print('Contrastive pretraining did not improve retrieval quality over ImageNet-only pretraining on this val set.')

## 10. Phase 2 Arm 3 — evidence fusion (joint attention, mirrors Arm 3c)

Reuses the joint-attention evidence-encoder design from Phase 1's Arm 3c, at `embed_dim=512` instead of 768 — a drop-in change. Defaults straight to the winning Phase 1 configuration (image+label joint attention) rather than re-testing label-only or early-fusion variants, since Phase 1 already established joint attention as best.

In [ ]:
# Train-vs-train retrieval, self-excluded by patient_id (study), NOT just by row
# index -- same fix as the Phase 1 notebook's Section 8. 85% of studies have 2+
# views sharing one report and one label vector; excluding only the query row
# left its study-mate in the pool (a neighbor whose label token IS the query's
# label), a regime val/test queries never see. The LEAKAGE CHECK line reports
# how often that actually fired.
train_pids_p2 = train_df['patient_id'].to_numpy()  # same row order as train_ds_p2 / train_img_p2
MAX_VIEWS_PER_STUDY = int(train_df.groupby('patient_id').size().max())
train_scores_raw_p2, train_neighbor_idx_raw_p2 = search(index_p2, train_img_p2, K + MAX_VIEWS_PER_STUDY)

train_neighbor_idx_p2, same_study_hits_p2 = [], 0
for i, row in enumerate(train_neighbor_idx_raw_p2):
    index_only = [j for j in row if j != i][:K]
    same_study_hits_p2 += int(any(train_pids_p2[j] == train_pids_p2[i] for j in index_only))
    keep = [j for j in row if j != i and train_pids_p2[j] != train_pids_p2[i]][:K]
    assert len(keep) == K, f'query {i}: only {len(keep)} non-self, non-same-study neighbors in the top {K + MAX_VIEWS_PER_STUDY}'
    train_neighbor_idx_p2.append(keep)
train_neighbor_idx_p2 = np.array(train_neighbor_idx_p2)
print('train_neighbor_idx_p2 shape:', train_neighbor_idx_p2.shape)
print(f'LEAKAGE CHECK: {same_study_hits_p2}/{len(train_pids_p2)} ({same_study_hits_p2 / len(train_pids_p2):.1%}) train queries had a '
      f'same-study view in their top-{K} under index-only exclusion -- now excluded.')

evidence_labels_train_p2 = train_lbl_p2[train_neighbor_idx_p2]
evidence_img_train_p2 = train_img_p2[train_neighbor_idx_p2]
evidence_labels_val_p2 = train_lbl_p2[neighbor_idx_p2]
evidence_img_val_p2 = train_img_p2[neighbor_idx_p2]
evidence_labels_test_p2 = train_lbl_p2[test_neighbor_idx_p2]
evidence_img_test_p2 = train_img_p2[test_neighbor_idx_p2]

train_ds_arm3_p2 = TensorDataset(torch.from_numpy(train_img_p2), torch.from_numpy(train_lbl_p2),
                                  torch.from_numpy(evidence_labels_train_p2), torch.from_numpy(evidence_img_train_p2))
val_ds_arm3_p2 = TensorDataset(torch.from_numpy(val_img_p2), torch.from_numpy(val_lbl_p2),
                                torch.from_numpy(evidence_labels_val_p2), torch.from_numpy(evidence_img_val_p2))
train_loader_arm3_p2 = DataLoader(train_ds_arm3_p2, batch_size=BATCH_SIZE, shuffle=True)
val_loader_arm3_p2 = DataLoader(val_ds_arm3_p2, batch_size=BATCH_SIZE, shuffle=False)
test_ds_arm3_p2 = TensorDataset(torch.from_numpy(test_img_p2), torch.from_numpy(test_lbl_p2),
                                 torch.from_numpy(evidence_labels_test_p2), torch.from_numpy(evidence_img_test_p2))
test_loader_arm3_p2 = DataLoader(test_ds_arm3_p2, batch_size=BATCH_SIZE, shuffle=False)
print('Phase 2 Arm 3 dataset sizes:', len(train_ds_arm3_p2), len(val_ds_arm3_p2), len(test_ds_arm3_p2))

In [ ]:
class EvidenceEncoderP2(nn.Module):
    """Joint attention (mirrors Phase 1's EvidenceEncoderV3), at embed_dim=512."""
    def __init__(self, num_labels, embed_dim, num_heads=4, image_embed_dim=None, max_k=16):
        super().__init__()
        image_embed_dim = image_embed_dim or embed_dim
        self.label_proj = nn.Linear(num_labels, embed_dim)
        self.image_proj = nn.Linear(image_embed_dim, embed_dim)
        self.modality_embed = nn.Embedding(2, embed_dim)
        self.neighbor_pos_embed = nn.Embedding(max_k, embed_dim)
        self.attention = nn.MultiheadAttention(embed_dim, num_heads=num_heads, batch_first=True)

    def forward(self, query_embedding, evidence_labels, evidence_image_embeddings):
        k = evidence_labels.shape[1]
        dev = evidence_labels.device
        label_embed = self.label_proj(evidence_labels)
        image_embed = self.image_proj(evidence_image_embeddings)
        pos = self.neighbor_pos_embed(torch.arange(k, device=dev)).unsqueeze(0)
        image_type = self.modality_embed.weight[0]
        label_type = self.modality_embed.weight[1]
        image_tok = image_embed + pos + image_type
        label_tok = label_embed + pos + label_type
        evidence_tokens = torch.cat([image_tok, label_tok], dim=1)
        query = query_embedding.unsqueeze(1)
        context, _weights = self.attention(query, evidence_tokens, evidence_tokens)
        return context.squeeze(1)


class FrozenDualTowerEvidenceClassifierP2(nn.Module):
    """No owned encoder -- query_embedding is precomputed/frozen, passed in
    directly. Mirrors src/models/frozen_fusion_classifier.py."""
    def __init__(self, num_labels, embed_dim=512, image_embed_dim=512, max_k=16):
        super().__init__()
        self.evidence_encoder = EvidenceEncoderP2(num_labels, embed_dim, image_embed_dim=image_embed_dim, max_k=max_k)
        self.head = nn.Linear(embed_dim, num_labels)

    def forward(self, query_embedding, evidence_labels, evidence_image_embeddings, evidence_confidence=None):
        evidence_context = self.evidence_encoder(query_embedding, evidence_labels, evidence_image_embeddings)
        if evidence_confidence is not None:
            fused = query_embedding + evidence_context * evidence_confidence.unsqueeze(-1)
        else:
            fused = query_embedding + evidence_context
        return self.head(fused)


set_seed(SEED)
model_arm3_p2 = FrozenDualTowerEvidenceClassifierP2(num_labels=len(label_vocab), image_embed_dim=512, max_k=K).to(device)
# Arm 3 LR: swept {1e-4, 3e-4, 1e-3} x 3 seeds on saved embeddings, selected on val: 1e-4 and 3e-4 tie
# (val 0.7247); 1e-3 uses the evidence harder (ablation shift 0.09 vs 0.01) and TEST gets worse (0.685 vs
# 0.697). It early-stops after ~12-40 epochs, so the longer PATIENCE_PROBE=50 budget is not needed to
# converge, just harmless.
optimizer_arm3_p2 = torch.optim.Adam(model_arm3_p2.parameters(), lr=LR)

best_auroc_arm3_p2, best_state_arm3_p2, epochs_without_improvement_arm3_p2 = -1.0, None, 0

for epoch in range(1, EPOCHS_PROBE + 1):  # cached embeddings, ~1.7 s/epoch -- same budget as the probes
    model_arm3_p2.train()
    total_loss = 0.0
    for query_emb, labels, ev_labels, ev_images in train_loader_arm3_p2:
        query_emb, labels = query_emb.to(device), labels.to(device)
        ev_labels, ev_images = ev_labels.to(device), ev_images.to(device)
        optimizer_arm3_p2.zero_grad()
        loss = criterion(model_arm3_p2(query_emb, ev_labels, ev_images), labels)
        loss.backward()
        optimizer_arm3_p2.step()
        total_loss += loss.item() * query_emb.size(0)
    total_loss /= len(train_ds_arm3_p2)

    model_arm3_p2.eval()
    all_labels3_p2, all_scores3_p2 = [], []
    with torch.no_grad():
        for query_emb, labels, ev_labels, ev_images in val_loader_arm3_p2:
            probs = torch.sigmoid(model_arm3_p2(query_emb.to(device), ev_labels.to(device), ev_images.to(device))).cpu().numpy()
            all_labels3_p2.append(labels.numpy())
            all_scores3_p2.append(probs)
    val_auroc3_p2 = macro_auroc(np.concatenate(all_labels3_p2), np.concatenate(all_scores3_p2))
    if epoch % 10 == 0 or epoch == 1:
        print(f'[Phase 2 Arm 3] epoch {epoch}/{EPOCHS_PROBE}  train_loss={total_loss:.4f}  val_macro_auroc={val_auroc3_p2:.4f}')

    if val_auroc3_p2 > best_auroc_arm3_p2:
        best_auroc_arm3_p2, best_state_arm3_p2, epochs_without_improvement_arm3_p2 = val_auroc3_p2, copy.deepcopy(model_arm3_p2.state_dict()), 0
    else:
        epochs_without_improvement_arm3_p2 += 1
        if epochs_without_improvement_arm3_p2 >= PATIENCE_PROBE:
            print(f'[Phase 2 Arm 3] no val improvement for {PATIENCE_PROBE} epochs -- stopping early at epoch {epoch}')
            break

model_arm3_p2.load_state_dict(best_state_arm3_p2)
print(f'best Phase 2 Arm 3 val_macro_auroc = {best_auroc_arm3_p2:.4f}')
model_arm3_p2.eval()
test_auroc_arm3_p2 = held_out_auroc(lambda b: model_arm3_p2(b[0].to(device), b[2].to(device), b[3].to(device)), test_loader_arm3_p2, label_pos=1)
print(f'Phase 2 Arm 3 TEST macro_auroc = {test_auroc_arm3_p2:.4f}')
torch.save({'model_state_dict': model_arm3_p2.state_dict(), 'label_vocab': label_vocab}, '/kaggle/working/phase2_arm3_best.pt')

In [ ]:
model_arm3_p2.eval()
real_scores_arm3_p2, zeroed_scores_arm3_p2 = [], []
with torch.no_grad():
    for query_emb, labels, ev_labels, ev_images in val_loader_arm3_p2:
        query_emb_d = query_emb.to(device)
        ev_labels_d, ev_images_d = ev_labels.to(device), ev_images.to(device)
        real_scores_arm3_p2.append(torch.sigmoid(model_arm3_p2(query_emb_d, ev_labels_d, ev_images_d)).cpu().numpy())
        zeroed_scores_arm3_p2.append(torch.sigmoid(model_arm3_p2(query_emb_d, torch.zeros_like(ev_labels_d), torch.zeros_like(ev_images_d))).cpu().numpy())

real_scores_arm3_p2 = np.concatenate(real_scores_arm3_p2)
zeroed_scores_arm3_p2 = np.concatenate(zeroed_scores_arm3_p2)
shift_arm3_p2 = np.mean(np.abs(real_scores_arm3_p2 - zeroed_scores_arm3_p2))
print(f'[Phase 2 Arm 3] evidence-ablation mean |shift| = {shift_arm3_p2:.4f}')
print()
# Arm 3's query embedding is IMAGE-ONLY (mirrors Phase 1's Arm 3c-frozen exactly,
# which also only ever had an image query) -- so the fair baseline for "does
# retrieval evidence help" is Arm 1's image-only ablation, not Arm 1's full
# image+text number. Comparing against auroc_full_p2 would be apples-to-oranges,
# since Arm 3 never sees text at all.
print('=== Phase 2: Arm 1 (image-only PROBE, no retrieval) vs. Arm 3 (image + retrieval evidence) ===')
print(f'Phase 2 Arm 1, dedicated image-only probe (no retrieval)  val_macro_auroc = {best_auroc_p2_img:.4f}   TEST = {test_imgprobe_p2:.4f}')
print(f'Phase 2 Arm 3 (image query + evidence fusion)             val_macro_auroc = {best_auroc_arm3_p2:.4f}   TEST = {test_auroc_arm3_p2:.4f}')
print(f'  evidence gain: val = {best_auroc_arm3_p2 - best_auroc_p2_img:+.4f}   TEST = {test_auroc_arm3_p2 - test_imgprobe_p2:+.4f}   (val is selected-on; TEST is the number to report)')
print(f'  (against the image-only ABLATION of the fused head, as earlier runs reported: val {best_auroc_arm3_p2 - auroc_image_only_p2:+.4f} / TEST {test_auroc_arm3_p2 - test_image_only_p2:+.4f} -- that baseline is a head tuned for image+text, so it flatters Arm 3)')
if test_auroc_arm3_p2 > test_imgprobe_p2:
    print(f'Retrieval evidence improves on the image-only probe by {test_auroc_arm3_p2 - test_imgprobe_p2:.4f} TEST AUROC -- '
          'consistent with Phase 1\'s finding (Arm 3c-frozen beat Arm 1).')
else:
    print(f'Retrieval evidence does not improve on the image-only probe on TEST ({test_imgprobe_p2 - test_auroc_arm3_p2:.4f} behind) -- '
          'contrasts with Phase 1\'s finding; check the evidence-ablation shift above (if it is ~0, Arm 3 is not using the evidence at all).')
print('(Not compared against auroc_full_p2 -- that number includes text, which Arm 3\'s query never sees;')
print('comparing against it would conflate "does evidence help" with "does text help", the two separate')
print('conclusions this project tracks. See Section 11 for the text conclusion.)')

## 11. Final comparison — both conclusions, explicit

In [ ]:
print('=== Conclusion 1: does retrieval-evidence fusion improve classification? ===')
print('(Arm 3\'s query embedding is image-only in both phases -- Phase 2 Arm 3 vs. Arm 1 compares')
print('against Arm 1\'s IMAGE-ONLY ablation, not its full image+text number, to keep this an')
print('apples-to-apples "does evidence help" test, not conflated with "does text help".)')
print(f'Phase 1  Arm 1 (image-only, ImageNet-pretrained)              val_macro_auroc = {PHASE1_ARM1_AUROC:.4f}')
print(f'Phase 1  Arm 3c-frozen (evidence fusion)                      val_macro_auroc = {PHASE1_ARM3C_FROZEN_AUROC:.4f}')
print(f'  -> Phase 1 gain from evidence: {PHASE1_ARM3C_FROZEN_AUROC - PHASE1_ARM1_AUROC:.4f}')
print(f'Phase 2  Arm 1, image-only probe (contrastively pretrained image)     val_macro_auroc = {best_auroc_p2_img:.4f}   TEST = {test_imgprobe_p2:.4f}')
print(f'Phase 2  Arm 3 (image query + evidence fusion)                       val_macro_auroc = {best_auroc_arm3_p2:.4f}   TEST = {test_auroc_arm3_p2:.4f}')
print(f'  -> Phase 2 gain from evidence: val = {best_auroc_arm3_p2 - best_auroc_p2_img:+.4f}   TEST = {test_auroc_arm3_p2 - test_imgprobe_p2:+.4f}')
print('  (Phase 1 reference numbers above are val-only -- the Phase 1 notebook now also reports TEST; rerun it for the like-for-like held-out comparison.)')
print()
print('=== Conclusion 2: does adding text improve decision-making? (three-way split) ===')
print(f'Phase 1  Arm 1 (image-only, ImageNet-pretrained)                         val_macro_auroc = {PHASE1_ARM1_AUROC:.4f}')
print(f'Phase 2  Arm 1, image-only probe (contrastively pretrained image)        val_macro_auroc = {best_auroc_p2_img:.4f}   TEST = {test_imgprobe_p2:.4f}')
print(f'Phase 2  Arm 1, full (contrastively pretrained image + text)            val_macro_auroc = {auroc_full_p2:.4f}   TEST = {test_full_p2:.4f}')
print(f'Phase 2  Arm 1, text-only probe                                         val_macro_auroc = {best_auroc_p2_txt:.4f}   TEST = {test_txtprobe_p2:.4f}')
print(f'  (ablations of the fused head, for continuity: image-only {auroc_image_only_p2:.4f}/{test_image_only_p2:.4f}, text-only {auroc_text_only_p2:.4f}/{test_text_only_p2:.4f} val/TEST)')
print('  -> gap 1 (pretraining objective alone, val vs Phase 1 val):', f'{best_auroc_p2_img - PHASE1_ARM1_AUROC:+.4f}')
print('  -> gap 2 (adding text, full vs image-only probe):', f'val = {auroc_full_p2 - best_auroc_p2_img:+.4f}   TEST = {test_full_p2 - test_imgprobe_p2:+.4f}')
print('  -> text-only probe vs full:', f'val = {best_auroc_p2_txt - auroc_full_p2:+.4f}   TEST = {test_txtprobe_p2 - test_full_p2:+.4f}   (positive = text alone beats fusion, the recurring signature)')
print()
print('=== For reference: Phase 2 attempt 1 (superseded, naive end-to-end fusion) ===')
print(f'full={PHASE2_ATTEMPT1_FULL_AUROC:.4f}  image_only={PHASE2_ATTEMPT1_IMAGE_ONLY_AUROC:.4f}  text_only={PHASE2_ATTEMPT1_TEXT_ONLY_AUROC:.4f}  '
      '(modality collapse: image_only barely above chance)')
collapse_resolved = auroc_image_only_p2 > PHASE2_ATTEMPT1_IMAGE_ONLY_AUROC + 0.1
print(f'Phase 2 attempt 2 image-only ablation now: {auroc_image_only_p2:.4f} -- '
      f'{"collapse resolved" if collapse_resolved else "inconclusive, re-check"}')
print()
print(f'label-vocab lexical overlap (circularity floor, unchanged by any of the above): {overlap:.1%}')
print('Text-input classification results above should be read alongside this number -- Stage 1 contrastive')
print('pretraining fixes modality collapse but does NOT by itself prove the circularity risk is resolved.')